# IT594 Project: Notebook 03, WANDS baselines (BM25, dense, hybrid)

Measures **relevance (NDCG@10)** and **constraint violation (color)** for each system, **separately**.
Needs `color_checker.py` in the same folder as this notebook.

## 1. Load data and define NDCG

In [1]:
import re, os, numpy as np, pandas as pd
from collections import Counter

PATH = "WANDS/dataset/"
products = pd.read_csv(PATH + "product.csv", sep="\t")
queries  = pd.read_csv(PATH + "query.csv",   sep="\t")
labels   = pd.read_csv(PATH + "label.csv",   sep="\t")

GAIN = {"Exact": 2, "Partial": 1, "Irrelevant": 0}
labels["gain"] = labels["label"].map(GAIN)
qrels = {q: dict(zip(g["product_id"], g["gain"])) for q, g in labels.groupby("query_id")}
qtext = dict(zip(queries["query_id"], queries["query"]))

def dcg(gains):
    return sum(g / np.log2(i + 2) for i, g in enumerate(gains))

def ndcg_at_k(ranked_ids, judged, k=10):
    gains = [judged.get(p, 0) for p in ranked_ids[:k]]        # unlabeled products count as 0
    ideal = sorted(judged.values(), reverse=True)[:k]
    return dcg(gains) / dcg(ideal) if dcg(ideal) > 0 else 0.0

print(products.shape, queries.shape, labels.shape)

(42994, 9) (480, 3) (233448, 5)


## 2. Product text used for retrieval
Choice to state in the report: **name + class + first 500 characters of description**.
`product_features` is deliberately **not** in the text, so we can measure whether systems respect attributes
they cannot directly read. Adding it later is a natural ablation.

In [2]:
for c in ["product_name", "product_class", "product_description"]:
    products[c] = products[c].fillna("")
products["text"] = products["product_name"] + ". " + products["product_class"] + ". " + products["product_description"].str.slice(0, 500)
pids = products["product_id"].values
products["text"].iloc[0][:300]

'solid wood platform bed. Beds. good , deep sleep can be quite difficult to have in this busy age . fortunately , there ’ s an antidote to such a problem : a nice , quality bed frame like the acacia kaylin . solidly constructed from acacia wood , this bed frame will stand the test of time and is fit '

## 3. Baseline 1: BM25 (lexical)

In [3]:
%pip install rank_bm25 -q
from rank_bm25 import BM25Okapi

tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([tok(t) for t in products["text"]])

def bm25_rank(q, k=100):
    s = bm25.get_scores(tok(q))
    return list(pids[np.argsort(-s)[:k]])

runs = {}
runs["bm25"] = {qid: bm25_rank(qtext[qid]) for qid in queries["query_id"]}
print("BM25 done")

Note: you may need to restart the kernel to use updated packages.
BM25 done


## 4. Baseline 2: dense retrieval (Sentence-Transformers)
Encodes products once (several minutes on CPU, cached to `wands_emb.npy`), then ranks by cosine similarity.

In [ ]:
%pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists("wands_emb.npy"):
    E = np.load("wands_emb.npy")
else:
    E = model.encode(products["text"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save("wands_emb.npy", E)

Qe = model.encode(queries["query"].tolist(), normalize_embeddings=True)
S = Qe @ E.T                                                   # cosine similarity (vectors are normalized)
runs["dense"] = {qid: list(pids[np.argsort(-S[i])[:100]]) for i, qid in enumerate(queries["query_id"])}
print("dense done")

## 5. Baseline 3: hybrid (Reciprocal Rank Fusion of BM25 and dense)

In [ ]:
def rrf(rank_lists, k=60, top=100):
    score = {}
    for rl in rank_lists:
        for r, p in enumerate(rl):
            score[p] = score.get(p, 0) + 1 / (k + r + 1)
    return [p for p, _ in sorted(score.items(), key=lambda x: -x[1])[:top]]

runs["hybrid_rrf"] = {q: rrf([runs["bm25"][q], runs["dense"][q]]) for q in queries["query_id"]}
print({name: len(r) for name, r in runs.items()})

## 6. Relevance: NDCG@10 (all 480 queries)

In [ ]:
def mean_ndcg(run, qids, k=10):
    return np.mean([ndcg_at_k(run[q], qrels.get(q, {}), k) for q in qids])

def judged_share(run, qids, k=10):
    return np.mean([np.mean([p in qrels.get(q, {}) for p in run[q][:k]]) for q in qids])

all_q = list(queries["query_id"])
rel = pd.DataFrame({
    "NDCG@10": {n: mean_ndcg(r, all_q) for n, r in runs.items()},
    "top10_judged_share": {n: judged_share(r, all_q) for n, r in runs.items()},
}).round(3)
rel

`top10_judged_share` is how many top-10 products have a human label. Unlabeled ones score 0, so a low share means NDCG is a lower bound.

## 7. Constraint violation: color
Uses the shared checker (`color_checker.py`). The product color comes from `color` or `primarycolor` in
`product_features`; the product name is the fallback.

In [ ]:
from color_checker import query_colors, color_check

def parse_features(s):
    d = {}
    if pd.isna(s):
        return d
    for pair in s.split("|"):
        if ":" in pair:
            k, v = pair.split(":", 1)
            d[k.strip().lower()] = v.strip()
    return d

products["feat"] = products["product_features"].apply(parse_features)
products["pcolor"] = products["feat"].apply(lambda d: d.get("color") or d.get("primarycolor") or "")
pcolor = dict(zip(products["product_id"], products["pcolor"]))
pname  = dict(zip(products["product_id"], products["product_name"]))

LENIENT = True      # main mode; set False for the strict ablation
color_qids = [q for q in all_q if len(query_colors(qtext[q], LENIENT)) > 0]
print("queries naming a color:", len(color_qids))
print([qtext[q] for q in color_qids][:15])

In [ ]:
def color_outcomes(run, qid, k=10, lenient=LENIENT):
    qcols = query_colors(qtext[qid], lenient)
    return [color_check(qcols, pcolor[p], pname[p], lenient) for p in run[qid][:k]]

def per_query_violation(run, k=10, lenient=LENIENT):
    rows = []
    for q in color_qids:
        out = color_outcomes(run, q, k, lenient)
        m, mm = out.count("match"), out.count("mismatch")
        rows.append({"qid": q,
                     "viol_rate": mm / (m + mm) if (m + mm) else np.nan,   # among checkable results
                     "viol_per_k": mm / k,                                  # fraction of top-k that clearly violate
                     "unresolved": (out.count("ambiguous") + out.count("unknown")) / k})
    return pd.DataFrame(rows).set_index("qid")

viol = {n: per_query_violation(r) for n, r in runs.items()}
constraint = pd.DataFrame({
    n: {"violation_rate (checkable)": v["viol_rate"].mean(),
        "violating share of top-10": v["viol_per_k"].mean(),
        "unresolved share": v["unresolved"].mean(),
        "NDCG@10 (color queries only)": mean_ndcg(runs[n], color_qids)}
    for n, v in viol.items()}).T.round(3)
constraint

**Report these two tables separately** (relevance and constraint). A system can win on NDCG and still lose on violation rate.
`unresolved share` is the part of the top-10 where the checker could not decide (missing or multicolor).

## 8. Is the difference real? Paired bootstrap over color queries

In [ ]:
def bootstrap_diff(a, b, col="viol_rate", n=5000, seed=0):
    d = (viol[a][col] - viol[b][col]).dropna().values
    rng = np.random.default_rng(seed)
    means = [rng.choice(d, len(d)).mean() for _ in range(n)]
    return d.mean(), np.percentile(means, [2.5, 97.5]), len(d)

for a, b in [("dense", "bm25"), ("hybrid_rrf", "bm25"), ("hybrid_rrf", "dense")]:
    m, ci, n = bootstrap_diff(a, b)
    print(f"{a:10s} - {b:10s}: violation-rate diff {m:+.3f}  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]  (n={n} queries)")

If the interval contains 0, you can't claim a difference. With only a few dozen color queries, wide intervals are expected, and ESCI can add power.

## 9. Look at actual violations (error analysis)

In [ ]:
def show_violations(system, n=12, k=10):
    rows = []
    for q in color_qids:
        for p, o in zip(runs[system][q][:k], color_outcomes(runs[system], q, k)):
            if o == "mismatch":
                rows.append({"query": qtext[q], "product": pname[p][:60], "color_field": pcolor[p],
                             "label": {2: "Exact", 1: "Partial", 0: "Irrelevant"}.get(qrels.get(q, {}).get(p), "unjudged")})
    df = pd.DataFrame(rows)
    print(system, "- total violations in top-10:", len(df))
    print(df["label"].value_counts().to_string())
    return df.sample(min(n, len(df)), random_state=1)

show_violations("dense")

## 10. Notes for the report
- State: unlabeled products count as 0 in NDCG; top-k judged share is reported.
- State: color checker v2.1 (lenient mode), validated on 200 labeled pairs (AI-labeled; disclose).
- Strict-mode ablation: set `LENIENT = False` and re-run sections 7 to 8.
- Hypothesis check: does dense have a higher violation rate than BM25 at equal or higher NDCG? Use the bootstrap in section 8.